# Evaluation pipeline

## Setup envoironment

In [1]:
import os, sys
os.chdir(os.path.dirname(sys.prefix))
print(f"Working directory: {os.getcwd()}")

Working directory: /home/trxxnk/mycode/diplom


## Import libs

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.transforms import v2

import mlflow.pytorch

import os
import cv2
import json
import mlflow
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
from src.tps_dewarp.dataset import TPSDataset, CanvasSpatialSpec
from src.models.tpsresnet18 import TPSResNet18
from src.tps_dewarp.geometry import build_remap_from_delta_tps

## Setup dugshub, mlflow, torch

In [11]:
import dagshub
dagshub.init(repo_owner='trxxnk', repo_name='text-image-alignment', mlflow=True)

Initialized MLflow to track repo "trxxnk/text-image-alignment"

Repository trxxnk/text-image-alignment initialized!

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Dataset

### Load Dataset

In [5]:
CANVAS_SIZE = 256
spatial_spec = CanvasSpatialSpec(CANVAS_SIZE, CANVAS_SIZE, mode="letterbox", fill=0.0)
photometric_transform = v2.Normalize(mean=[0.5], std=[0.5])

In [6]:
DATASET_DIR = "data/generated/v3"
dataset_with_meta = TPSDataset(
    DATASET_DIR,
    spatial_spec=spatial_spec,
    photometric_transform=photometric_transform,
    return_meta=True,
    lru_cache_maxsize=15,
)
len(dataset_with_meta)

33428

## Eval example

### Load model

In [12]:
model_name = "model_20260516_050857_epoch42"
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.pytorch.load_model(model_uri, map_location=device)

2026/05/17 16:25:32 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.11.0+cpu'


In [14]:
model

TPSResNet18(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, trac

In [15]:
N = 3337
img, tps, difficulty, meta = dataset_with_meta[N]
meta

{'original': 'raw_18113.png',
 'warped': 'raw_18113_medium.png',
 'is_identity': False}

In [16]:
def denorm_canvas_to_u8(img_t: torch.Tensor) -> np.ndarray:
    """(1,H,W) после Normalize(0.5,0.5) -> uint8 HW для OpenCV."""
    x = img_t.squeeze(0) * 0.5 + 0.5
    return (x * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()


N = 3337
img, tps, difficulty, meta = dataset_with_meta[N]
warped_u8 = denorm_canvas_to_u8(img)

map_x, map_y = build_remap_from_delta_tps(
    delta_tps=tps.numpy(),
    H=CANVAS_SIZE,
    W=CANVAS_SIZE,
    grid_size=9,
    clip=False,
)

rectified = cv2.remap(
    warped_u8,
    map_x,
    map_y,
    interpolation=cv2.INTER_CUBIC,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=255,
)

cv2.imwrite("out.png", rectified)

True

In [17]:
N = 3337
img, tps, difficulty, meta = dataset_with_meta[N]
meta

{'original': 'raw_18113.png',
 'warped': 'raw_18113_medium.png',
 'is_identity': False}

In [18]:
img = Image.open("data/generated/v3/raw_18113_medium.png").convert("L")

In [19]:
# ---------- INFERENCE (вход = тот же канон, что при обучении) ----------
N = 3337
img_c, tps_gt, _, _ = dataset_with_meta[N]
with torch.no_grad():
    pred = model(img_c.unsqueeze(0).to(device)).detach().cpu().numpy().reshape(9 * 9, 2)
pred[:5]

array([[0.0013, 0.0013],
       [0.0013, 0.0013],
       [0.0014, 0.0012],
       [0.0014, 0.0012],
       [0.0014, 0.0012]], dtype=float32)

In [20]:
warped_u8 = denorm_canvas_to_u8(img_c)
map_x, map_y = build_remap_from_delta_tps(
    delta_tps=pred,
    H=CANVAS_SIZE,
    W=CANVAS_SIZE,
    grid_size=9,
    clip=False,
)

rectified = cv2.remap(
    warped_u8,
    map_x,
    map_y,
    interpolation=cv2.INTER_CUBIC,
    borderMode=cv2.BORDER_CONSTANT,
    borderValue=255,
)

cv2.imwrite("out_model.png", rectified)

True

In [21]:
tps[:5]

tensor([[0.0000, 0.0000],
        [0.0000, 0.0107],
        [0.0000, 0.0193],
        [0.0000, 0.0241],
        [0.0000, 0.0241]])